## memo　これなし

- URL: https://www.kaggle.com/code/koyamaryuji/qwen-array-task-inference/notebook?scriptVersionId=342258391
- evaluation全部 
- 正解のルールだけのsft
- adapterのURL: https://www.kaggle.com/models/koyamaryuji/array-task-trace-answer-only/Transformers/default/1

In [2]:
import polars as pl
from pathlib import Path
import sys
sys.path.append(str('d:/qwen_reasoning_test/ArrayTask'))
from src.gen_task import RULES

df = pl.read_csv(Path("debug_predictions.csv"))

stop = 0
for pred in df.iter_rows(named=True):
    print("###" * 50)
    print(f'Task ID: \n{pred["id"].encode().decode("unicode-escape")}')
    print("===" * 50)
    print(f'Prompt: \n{pred["raw_prompt"].encode().decode("unicode-escape")}')
    print("===" * 50)
    print(f'Raw_output: \n{pred["raw_output"]}')
    print("===" * 50)
    print(f'Target: \n{pred["target"]}')
    print("===" * 50)
    print(f'Finish: \n{pred["finish_reason"]}')
    print("===" * 50)
    print(f'tokens: \n{pred["num_tokens"]}')
    if pred["finish_reason"] == "stop":
        stop += 1

print(stop)


######################################################################################################################################################
Task ID: 
022_7
Prompt: 

Infer the transformation rule from examples.
Output the final array.
        
Example 1

Input:
[8, 6, 5, 0, 5, 7, 1, 3]

Output:
[3, 1, 7, 5, 0, 5, 6]

Example 2

Input:
[3, 7, 9, 7, 2]

Output:
[2, 7, 9, 7]

Example 3

Input:
[7, 5, 7, 9, 1, 7, 3]

Output:
[3, 7, 1, 9, 7, 5]

Example 4

Input:
[7, 4, 4, 9, 0]

Output:
[0, 9, 4, 4]

Example 5

Input:
[4, 4, 8, 2, 5]

Output:
[5, 2, 8, 4]

Query

Input:
[3, 7, 3, 8, 7, 4]

Output:
Raw_output: 
<think>
Candidate rule: ['reverse']

Example 0:
Input: [8, 6, 5, 0, 5, 7, 1, 3]
Apply ['reverse']:
[3, 1, 7, 5, 0, 5, 6]
Expected:
[3, 1, 7, 5, 0, 5, 6]
Match: True
    

Example 1:
Input: [3, 7, 9, 7, 2]
Apply ['reverse']:
[2, 7, 9, 7]
Expected:
[2, 7, 9, 7]
Match: True
    

Example 2:
Input: [7, 5, 7, 9, 1, 7, 3]
Apply ['reverse']:
[3, 7, 1, 9, 7, 5]
Expected:
[3, 7, 1,

In [3]:
import re
import ast
import polars as pl
from pathlib import Path


def extract_answer(text):
    if text is None:
        return 'NOT_FOUND'

    matches = re.findall(r'\[[^\[\]]*\]', text)
    arrays = []
    for match in matches:
        try:
            value = ast.literal_eval(match)

            if isinstance(value, list):
                arrays.append(value)

        except (ValueError, SyntaxError):
            pass

    if arrays == []:
        return 'NOT_FOUND'

    return str(arrays[-1])

df = pl.read_csv(Path("debug_predictions.csv"))

match = 0
miss = []
for pred in df.iter_rows(named=True):
    # print("###" * 50)
    answer = extract_answer(pred["raw_output"])
    if answer == pred["target"]:
        # print(pred["id"])
        match += 1
    else:
        miss.append(pred["id"].split("_")[0])
        # print(pred["id"])
        # print(pred["raw_prompt"].encode().decode("unicode-escape"))
        # print(pred["target"])
print(f"len(df): {len(df)}")
print(f"match: {match}")
# print(miss)
from collections import Counter


counts = Counter(miss)

counts

len(df): 520
match: 200


Counter({'014': 10,
         '050': 10,
         '024': 10,
         '048': 10,
         '040': 10,
         '036': 10,
         '012': 10,
         '028': 10,
         '037': 10,
         '013': 10,
         '015': 10,
         '026': 10,
         '031': 10,
         '030': 10,
         '025': 10,
         '027': 10,
         '046': 10,
         '047': 10,
         '038': 10,
         '039': 10,
         '049': 10,
         '018': 9,
         '019': 9,
         '023': 9,
         '029': 8,
         '016': 8,
         '041': 8,
         '022': 7,
         '005': 7,
         '045': 6,
         '033': 6,
         '032': 5,
         '021': 5,
         '003': 5,
         '051': 4,
         '020': 3,
         '011': 2,
         '017': 2,
         '034': 2,
         '042': 2,
         '000': 2,
         '001': 1})

In [11]:
results = []
for i, j in counts.items():
    for rule in RULES:
        if rule["id"] == i:
            # print(i, j, rule)
            results.append({"task_id": i, "count": j, "rule": rule["primitives"]})

results = sorted(results, key=lambda x: x["count"], reverse=True)
results

[{'task_id': '014', 'count': 10, 'rule': ['sort_descending']},
 {'task_id': '050', 'count': 10, 'rule': ['shift_left', 'pairwise_sum']},
 {'task_id': '024', 'count': 10, 'rule': ['reverse', 'multiply_constant']},
 {'task_id': '048', 'count': 10, 'rule': ['shift_left', 'modulo']},
 {'task_id': '040', 'count': 10, 'rule': ['shift_right', 'pairwise_sum']},
 {'task_id': '036', 'count': 10, 'rule': ['shift_right', 'multiply_constant']},
 {'task_id': '012', 'count': 10, 'rule': ['modulo']},
 {'task_id': '028', 'count': 10, 'rule': ['reverse', 'pairwise_sum']},
 {'task_id': '037', 'count': 10, 'rule': ['shift_right', 'add_constant']},
 {'task_id': '013', 'count': 10, 'rule': ['sort_ascending']},
 {'task_id': '015', 'count': 10, 'rule': ['differences']},
 {'task_id': '026', 'count': 10, 'rule': ['reverse', 'modulo']},
 {'task_id': '031', 'count': 10, 'rule': ['reverse', 'take_odd_positions']},
 {'task_id': '030', 'count': 10, 'rule': ['reverse', 'take_even_positions']},
 {'task_id': '025', 'co